Join between candidate and italian_cities
on cap
9/436 = 2% old cap

Join between candidate_cleaned and italian_cities
on cap
9/436 = 2% old cap

Join between candidate_cleaned and italian_cities
on city
1/436 = 0.2% city mismatch

---------

Join between joborder and italian_cities
on cap
1209/35898 = 3.3% old cap

Join between joborder_cleaned and italian_cities
on cap
895/35898 = 2.5% old caps

Join between joborder_cleaned and italian_cities
on city
728/35898 = 2% city mismatch



Ho una tabella candidato CA con info riguardo cap, città, provincia regione. questi campi potrebbero essere sbagliati. ho anche una tabella con tutti i comuni italiani CI e le stesse inso, questi sono sicuramente corretti.

Storicamente in Italia: dato un indirizzo è piu probabile che questo cambia cap o che cambia nome del comune? 
devo decidere se utilizzare CA.cap come lookup in CI e sostituire CA.città con CI.città (se presente), o se utilizzare CA.città come lookup in CI e sostituire CA.cap con CI.cap (se presente)

Soluzione:
i cambi di nome sono molto più rari rispetto a quelli dei cap. Quindi utilizza city (ed eventualmente province come lookup)

I have a candidate table CA which has the following columns:
    - cap
    - city
    - province
    - region
This table may have wrong data. 

I also have a italian_cities table IC. This table has a row for each pair (cap, city), and the following columns:
    - cap
    - city
    - province
    - region
    - lat
    - lng

Regarding the wrong data, these are all the cases:
    - a cap/city is valid (it's avaiable in CA and there is a match in IC)
    - a cap/city is wrong (it's available in CA but there is no match in IC)
    - a cap/city is not available in CA

I want to use IC to fill in the wrong/missing information in CA.

I'll have to work with (cap, city) pairs, because they uniquely identify coordinates that will allow me to compute distance between candidates.
Notice that:
    - a city may have multiple caps. those are always the biggest cities, hence most candidates will be located there.
    - a cap may be assigned to multiple cities. This happens for small village, therefore only a small population of candidates will be located there.

Even when a cap/city is valid, I don't know whether it's correctly assigned to a candidate or not. So I would like to choose one as ground truth and correct the other. Considering the statistics above, I'll chooose cap as ground truth. Is it a correct choise or I'd rather choose city?

For cap I would like to use the following logic:
    - if CA.cap is valid keep it.
    - otherwise, if the CA.city is valid take the corresponding IC.CAP if IC.CAP is unique
    - otherwise null

For city, instead, I would like to use the following logic:
    - if CA.cap is valid, take all the IC.city with that cap: if CA.city belongs to one of those then keep CA.city
    - otherwise if there is only one IC.city associated to CA.cap, take IC.city
    - otherwise null

I would probably keep the old cap, city and store the resolved cap/city in new columns

Does this logic make sense?

------------------------

So far the IC table contain a row for each valid (cap, city) pair.
But while some candidates have both valid cap and city, others may only one of them

Because of this, I would add new rows in the italian_cities table:
    1. (Null, city) one row for each city that has multiple caps: that city will have (lat, lng) equal to the average (lat, lng) for eacy pair (?, CA.city)
    2. (cap, Null) a row for each cap that is shared among multiple cities: that cap will have (lat, lng) equal to the average (lat, lng) for eacy pair (CA.cap, ?)
and also add a location_type which will be either EXACT, CAP_ONLY, CITY_ONLY.

Case 1. will be useful to fetch (lat, lng) of a candidate with invalid cap and valid city, while that city does not have a unique cap
Case 2. will be useful to fetch (lat, lng) of candidate with valid cap and invalid city, while that cap is assigned to multiple cities


------------------------

Then I would create a new table ICM with a row for each pair of (cap, city) in IC, with the precomputed distance in km between them. It will have colums:
    - from_cap
    - from_city
    - to_cap
    - to_city
    - distance

------------------------

Finally I would create a new table CAM with a row for each pair of candidates that are within X chilometers nearby.
I will use ICM to get the distance between candidates.




I have a italian_cities table IC in Snowflake. This table has a row for each pair (cap, city), and the following columns:
    - cap
    - city
    - province
    - region
    - lat
    - lng

Notice that:
    - a city may have multiple caps. so there will be multiple rows with the same city and different caps. those are always the biggest cities, hence most candidates will be located there. 
    - a cap may be assigned to multiple cities. so there will be multiple rows with same cap and different city. This happens for small village, therefore only a small population of candidates will be located there.

I would add new rows in the italian_cities table:
    1. row for each city that has multiple caps, add a row with (Null, city): that row will have (lat, lng) equal to the average (lat, lng) for eacy pair (?, CA.city) in the dataset
    2. row for each cap that is shared among multiple cities, add a row (cap, Null): that row will have (lat, lng) equal to the average (lat, lng) for eacy pair (CA.cap, ?)

For the other columns, take them only if all rows have the same value. For example, if the region of all cities corresponding to the same cap are the same, take the region, otherwise set it to null.

and also add a location_type column which will be either EXACT, CAP_ONLY, CITY_ONLY.



---------------------------------------------------------------------------------------

I have a candidate table CA which has the following columns:
    - cap
    - city
    - province
    - region
This table may have wrong data. 

I also have a italian_cities table IC. This table has a row for each pair (cap, city), and the following columns:
    - cap
    - city
    - province
    - region
    - lat
    - lng

Regarding the wrong data, these are all the cases:
    - a cap/city is valid (it's avaiable in CA and there is a match in IC)
    - a cap/city is wrong (it's available in CA but there is no match in IC)
    - a cap/city is not available in CA

I want to use IC to fill in the wrong/missing information in CA.

I'll have to work with (cap, city) pairs, because they uniquely identify coordinates that will allow me to compute distance between candidates.
Notice that:
    - a city may have multiple caps. those are always the biggest cities, hence most candidates will be located there.
    - a cap may be assigned to multiple cities. This happens for small village, therefore only a small population of candidates will be located there.

Even when a cap/city is valid, I don't know whether it's correctly assigned to a candidate or not. So I would like to choose one as ground truth and correct the other. Considering the statistics above, I'll chooose cap as ground truth.

For resolved_cap I would like to use the following logic:
    - if CA.cap is valid keep it.
    - otherwise, if the CA.city is valid take the corresponding IC.CAP if IC.CAP is unique
    - otherwise null

For resolved_city, instead, I would like to use the following logic:
    - if CA.resolved_cap is valid, take all the IC.city with that cap: if CA.city belongs to one of those then keep CA.city
    - otherwise if there is only one IC.city associated to CA.resolved_cap, take IC.city
    - otherwise if CA.resolved_cap is null, and CA.city is a valid city, keep it


I would probably keep the old cap, city and store the resolved cap/city in new columns



CAP	    CITY	        RESOLVED_CAP	RESOLVED_CITY
40012	bologna	        40012	        calderara di reno   -> 40012 is associated to Calderara di Reno, not Bologna
39011	lana lana	    39011	        lana                -> 39011 is associated to Lana, not Lana Lana (which does not exist)
40056	valsamoggia	    40053	        valsamoggia         -> 40056 does not exist, instead 40056 is the cap associated (uniquely) to Valsamoggia
57100	livorno		    Null            livorno             -> 57100 is an old generic cap for Livorno (which now has multiple caps)
80020	napoli	        80020           Null                -> 80020 is associated to a few small villages in the Napoli province, not Napoli
46020	borgo mantovano	46020	        NUll                -> 46020 is associated to a few small villages, not Borgo Mantovano (which is a placeholder for a generic village)




I have a candidate table CA with resolved_cap, resolved_city, province.
I have a joborder table CA with resolved_cap, resolved_city, region.
I have a italian_cities table IC with one row for each unique (cap, city) pair. it also has province, province_ext, region, lat, lng:
    - multiple rows may share the same cap (small villages are assigned the same cap)
    - multiple rows may share the same city (larger cities have multiple caps)

I want to get the coordinates from IC table for the candidate/joborder tables based on CA.cap_resolved and CA.city_resolved:
    - if resolved_cap and resolved_city match IC.cap and IC.city: take the corresponding IC.lat and IC.lng
    - if only one among resolved_cap and resolved_city is non null: get the average IC.lat and IC.lng correspoding to that cap/city locations
    - otherwise Null

For candidate, set resolved_province and resolved_region as follows:
    - if resolved_cap and resolved_city match IC.cap and IC.city: take the corresponding IC.province and IC.region
    - if only one among resolved_cap and resolved_city matches:
        - if the locations corresponding to resolved_cap/resolved_city all share the same province/region, take IC.province/IC.region
        - otherwise if CA.province corresponds to at least one of the IC.province, take that IC.province and IC.region
    - otherwise if CA.province/CA.region is valud, take them

For joborder, set resolved_province and resolved_region as follows:
    - if resolved_cap and resolved_city match IC.cap and IC.city: take the corresponding IC.province and IC.region
    - if only one among resolved_cap and resolved_city matches:
        - if the locations corresponding to resolved_cap/resolved_city all share the same province/region, take IC.province/IC.region
        - otherwise if CA.region corresponds to at least one of the IC.region, take that IC.province and IC.region
    - otherwise if CA.province/CA.region is valud, take them

REGION              RESOLVED_CAP    CITY_RESOLVED   RESOLVEC_PROVINCE   RESOLVED_REGION
trentino alto adige	38062	        arco	        tn	                trentino alto adige sudtirol    -> The region is somehow correct, but special regions need multilingual names
campania	        20143	        milano	        mi	                lombardia                       -> Milan is in Lombardia, not Campania
alessandria	        15067	        novi ligure	    al	                piemonte                        -> Alessandria is Novi Ligure's province, not it's region. It's region is Piemonte 

How to check that it properly works:




In [ ]:
I also have a italian_cities table IC. This table has a row for each pair (cap, city), and the following columns:
    - cap
    - city
    - province
    - region
    - lat
    - lng

Notice that:
- a city may have multiple caps. so there will be multiple rows with the same city and different caps. those are always the larger cities.
- a cap may be assigned to multiple cities. so there will be multiple rows with same cap and different city. This happens for small villages.

For this reason the IC table also contains, other than the usual (cap, city) rows, the following two type of rows:
(NULL, city):  a generic location that identifies a city with multiple caps
(cap, NULL):  a generic location that identifies a cap shared with multiple villages

This is also useful to match a candidate or a job position that has a cap but not a city, and viceversa.
In both cases, lat and lng are derived as the average value between all rows sharing the same city or cap

Objective:
I want to create a distance table DT that contains a row for each pair or rows in IC, and the following columns:
    - from_cap
    - from_city
    - to_cap
    - to_city
    - distance (in km)

I also want to filter out all rows in DT whose from/to locations are not within 100KM radius distance.
This table will be used later on to match candidates that are nearby job locations

Consider that is will be computed once a day
Do it in snowpark, provide the complete functions with all the improvements already




In [ ]:
I have a candidate table CA and a joborder table JO, with the following columns:
    - candidate_id_bh/joborder_id_bh
    - resolved_cap
    - resolved_city
    - job_site_cap
    - job_site_city

I have a candidate table CA and a joborder table JO, with the following columns:
    - candidate_id_bh/joborder_id_bh
    - resolved_cap
    - resolved_city

Both resolved_cap and resolved_city may be Null

I have a location_distance table LD, with the following columns:
    - from_cap
    - from_city
    - to_cap
    - to_city
    - distance (in km)
It may contain the following type of rows:
    (cap, city): a specific location
    (Null, city):  a generic location that identifies a city with multiple caps
    (cap, Null):  a generic location that identifies a cap shared with multiple villages


I want to create a match column between candidate and joborder. 
It will contain row for each (candidate, joborder) pair whose locations match a row in the LD table (which means candidate and joborder are within 100km)
I would like it to have the following columns:
    candidate_id_bh
    joborder_id_bh
    distance
